In [ ]:
############################################## SETTINGS ##############################################  
# settings
sim = 'sim1001'  # current steps for firing curve control
# sim = 'sim1002'  # current steps for firing curve 20% KCNQ
# sim = 'sim1003'  # current steps for firing curve control exact rheobase
# sim = 'sim1004'  # current steps for firing curve 20% KCNQ exact rheobase

cell_type='dspn'
model = 3

import os, sys
# compute absolute path of main folder
cwd = os.getcwd()

if os.path.exists(os.path.join(cwd, 'settings.py')):
    main_dir = cwd
else:
    main_dir = os.path.abspath(os.path.join(cwd, '..'))

# set CWD or add to path
os.chdir(main_dir)
sys.path.insert(0, main_dir)

# then import/run settings
%run -i settings.py

from analysis_functions import *

# Get the current working directory
current_wd = os.getcwd()
# 'sim' + str(sim)

downsample = True

home = os.path.expanduser('~')

# Set working directory
external = False
external_name = 'MacOS10'
target = f'model{model}'

if not external:
    if downsample:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'
else:
    if downsample:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'


# create path to directory to save analysis
base_path = os.path.join(home, 'Documents', 'Repositories', 'msNEURON_Belal2026', 'analysis')
sim_image_path = os.path.join(base_path, cell_type, sim)
os.makedirs(sim_image_path, exist_ok=True)

save = False

In [ ]:
# calculate surface area of a the neuron
# note NEURON’s default spatial units are micrometres (µm) so area returns µm²

cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=None)
area, total_length = surface_area(cell=cell, spines=spines)

# draw cell

cell_coordinates = []

for sec in cell.somalist:
    h('access ' + sec.name())
    x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
    cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])

# Setup for dendritic sections
for sec in cell.dendlist:
    for seg in sec:
        x, y, z, diam = interpolate_3d(sec, seg.x)
        cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])
    
cell_coordinates = np.array(cell_coordinates, dtype=object)

    
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=0.8, s=2, color='grey', height=600, width=600, plane='xy', mirror=False)

fig_morphology.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    fig_morphology.write_image(f"{sim_image_path}/morphology.svg", format='svg', scale=1)


In [ ]:
# use load_data_dicts to load data
sims_dir = os.path.join(wd, sim)
data_dict = load_data_dicts(wd, sim, cell_type, verbose=True)

In [ ]:
# check memory usage
report_memory(verbose=True)

In [ ]:
# basic experimental metadata
ds = data_dict['metadata']['ds']
dt = data_dict['metadata']['dt']
dt = ds * dt

sim_time = data_dict['metadata']['sim_time']
step_end = sim_time - 200
step_duration = 2000
step_start = step_end - step_duration
step_current = -10

if sim in ['sim1001', 'sim1002']:
    current_step_range = list(range(80, 401, 10)) 
elif sim in ['sim1003', 'sim1004']:
    if cell_type == 'dspn':
        current_step_range = list(range(153, 158, 1))  # ~155 pA    
    elif cell_type == 'ispn':
        current_step_range = list(range(128, 133, 1))  # ~129 pA

mechs2extract = ['kaf', 'kas', 'kir', 'kcnq', 'sk', 'bk']

print(step_start, step_duration, step_end)


In [ ]:
# generate plots
x = np.arange(0,len(data_dict['vsoma'][0])*dt, dt)
if sim in ['sim1001', 'sim1002']:
    _range_subset = [i for i in range(0, len(current_step_range), 2)]
elif sim in ['sim1003', 'sim1004']:
    _range_subset = range(0, len(current_step_range)+1)


width_mm = 110
height_mm = 90

width = width_mm / 25.4
height = height_mm / 25.4

plot9_MLP(x=x, ydict=data_dict['vsoma'], _range=current_step_range, _range_subset=_range_subset, yabline=[-85, -60], ds=1,
          xaxis_range=[step_start-100, sim_time], yaxis_range=[-90, 40], sim_image_path=sim_image_path, 
          save_name='fig_summary1.svg', width=width, height=height, save=save)


In [ ]:
# generate plots
x = np.arange(0,len(data_dict['vsoma'][0])*dt, dt)
if sim in ['sim1001', 'sim1002']:
    if cell_type == 'ispn':
        _range_subset=[3, 6, 12]
    else:
        _range_subset=[3, 9, 12]   
elif sim in ['sim1003', 'sim1004']:
    if cell_type == 'ispn':
        _range_subset=[2]
    else:
        _range_subset=[3]   
    

plot9_MLP(x=x, ydict=data_dict['vsoma'], _range=current_step_range, _range_subset=_range_subset, yabline=[-85, -60], ds=1,
          xaxis_range=[step_start-100, sim_time], yaxis_range=[-95, 40], sim_image_path=sim_image_path, 
          save_name='fig_summary2.svg', width=width, height=height, save=save)


In [ ]:
for mech in mechs2extract:
    plot_mech_current_MLP(mech=mech, data_dict=data_dict, _range=current_step_range, _range_subset=_range_subset,
                      sim_image_path=sim_image_path, sim_time=sim_time, step_start=step_start,  ds=1, 
                      width=width, height=height, dt=dt, save=False)

In [ ]:
if sim in ['sim1001', 'sim1002']:

    from scipy.optimize import least_squares
    from matplotlib.ticker import MultipleLocator, FixedLocator
    
    lw = 0.8
    width_mm = 90
    height_mm = 90
    marker_size = 3
    text_color = 'grey'
    
    if sim in ['sim1002', 'sim1004']:
        open_circles = True
    else:
        open_circles = False
        
    ydict = data_dict['vsoma']
    dt_spike = deltat if 'deltat' in globals() else dt
    spike_counts = [count_spikes(np.asarray(trace).squeeze(), dt=dt_spike) for trace in ydict]
    
    I = np.asarray(current_step_range, float)
    spikes = np.asarray(spike_counts, float)
    rate = spikes / float(step_duration)
    
    def mod(p, x):
        I0, rmax, I50, n = p
        z = np.clip(x - I0, 0, None)
        return rmax * (z**n) / (z**n + I50**n + 1e-12)
    
    def resid(p):
        yhat = mod(p, I)
        return (yhat - rate)
    
    I_pos = I[rate > 0]
    I0_init = np.percentile(I_pos, 10) if I_pos.size else np.min(I)
    rmax_init = max(rate.max(), 1.0)
    I50_init = max(np.percentile(np.clip(I - I0_init, 0, None), 60), 1e-6)
    n_init = 2.0
    p0 = np.array([I0_init, rmax_init, I50_init, n_init])
    lb = np.array([np.min(I)-50.0, 0.0, 1e-6, 0.5])
    ub = np.array([np.max(I), rate.max()*10 + 100.0, (np.max(I)-np.min(I))*5 + 1000.0, 6.0])
    
    res = least_squares(resid, p0, bounds=(lb, ub), max_nfev=2000)
    p = res.x
    
    I_fit = np.linspace(I.min(), I.max(), 300)
    rate_fit = mod(p, I_fit)
    
    fig, ax = plt.subplots(figsize=(width_mm/25.4, height_mm/25.4), dpi=150)
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)
    
    # Plot the fit line behind the points:
    if open_circles:
        ax.plot(I_fit, rate_fit * step_duration, color=text_color, lw=lw,
            linestyle='--', label='Hill fit', zorder=1)
    
        ax.plot(I, spikes, 'o', color=text_color, markersize=marker_size,
                markerfacecolor='none', markeredgecolor=text_color, label='Spikes', zorder=2)
    else:
        ax.plot(I_fit, rate_fit * step_duration, color=text_color, lw=lw,
            linestyle='-', label='Hill fit', zorder=1)
        
        ax.plot(I, spikes, 'o', color=text_color, markersize=marker_size,
                label='Spikes', zorder=2)
    
    ax.set_xlim(np.min(I), np.max(I))
    ax.set_ylim(-1, 100)
    ax.set_xlabel('Current Step (pA)', fontsize=10, color=text_color)
    ax.set_ylabel('Spike Count', fontsize=10, color=text_color)
    ax.set_title('Spike Count vs Current Step', fontsize=12, color=text_color)
    
    ax.spines['left'].set_position(('outward', 5))
    ax.spines['bottom'].set_position(('outward', 5))
    ax.spines['left'].set_color(text_color)
    ax.spines['bottom'].set_color(text_color)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    start_tick = 100
    end_tick = np.max(I)
    ax.set_xticks(np.arange(start_tick, end_tick + 50, 50))
    ax.xaxis.set_minor_locator(MultipleLocator(25))
    xmin = np.min(I)
    ax.set_xlim(xmin - 3, end_tick + 3)
    
    ax.yaxis.set_major_locator(FixedLocator([0, 20, 40, 60, 80, 100]))
    ax.yaxis.set_minor_locator(FixedLocator([10, 30, 50, 70, 90]))
    ax.tick_params(axis='y', which='major', length=4, width=lw, colors=text_color)
    ax.tick_params(axis='y', which='minor', length=2, width=lw, colors=text_color)
    
    ax.tick_params(direction='out', width=lw, color=text_color, labelcolor=text_color, labelsize=8)
    ax.tick_params(which='major', length=4, width=lw, colors=text_color)
    ax.tick_params(which='minor', length=2, width=lw, colors=text_color)

    leg = ax.legend(frameon=False, loc='best')
    for txt in leg.get_texts():
        txt.set_color(text_color)

    fig.tight_layout()
    
    plt.rcParams['svg.fonttype'] = 'none'
    
    if save:
        save_dir = sim_image_path if sim_image_path is not None else '.'
        os.makedirs(save_dir, exist_ok=True)
        fig.savefig(f"{save_dir}/fig_spike_count_vs_steps_summary.svg", format='svg', dpi=300, transparent=True)
    
    
    # plt.close(fig)


In [ ]:
data_dict['capacitance'][0]

In [ ]:
# # create a html version
# if save:
#     from pathlib import Path

#     nb_name = "passive tune analysis Matplotlib.ipynb"
#     nb_path = Path(main_dir) / "analysis notebooks" / nb_name   # adjust folder if needed

#     output_dir = Path(main_dir) / "docs" / cell_type
#     output_dir.mkdir(parents=True, exist_ok=True)

#     if not nb_path.exists():
#         raise FileNotFoundError(f"Notebook not found: {nb_path}")

#     !jupyter nbconvert "{nb_path}" --to html --output-dir="{output_dir}"
